
# Phase 5-3: PU-Learning 모델 학습 (XGBoost 버전)
이 노트북은 07번 노트북의 Random Forest 베이스 모델을 **XGBoost**로 교체하여 
동일한 동구(Dong-gu) 지역의 극단적 불균형 데이터를 학습합니다.
훈련된 모델은 `models/universal_xgb_model.pkl`에 저장됩니다.


In [1]:

import geopandas as gpd
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

base_dir = Path('../../')
dataset_path = base_dir / 'data/processed/dataset_dong-gu_gwangju_south_korea_10m_buf10m_baseline.gpkg'
model_dir = base_dir / 'data/models'
model_dir.mkdir(parents=True, exist_ok=True)

print("✅ 라이브러리 및 경로 설정 완료")


✅ 라이브러리 및 경로 설정 완료


In [2]:
from sklearn.model_selection import train_test_split
print(f"📦 데이터셋 로드 중... ({dataset_path.name})")
gdf = gpd.read_file(dataset_path)

gdf['is_trash'] = gdf['is_trash'].fillna(0).astype(int)

feature_cols = [col for col in gdf.columns if '_count_' in col]
print(f"추출된 피처 목록: {feature_cols}")

X_df = gdf[feature_cols]
y = gdf['is_trash'].values

X_train, X_test, y_train, y_test = train_test_split(X_df, y, test_size=0.2, stratify=y, random_state=42)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Train Positive: {sum(y_train==1):,}, Test Positive: {sum(y_test==1):,}\n")


📦 데이터셋 로드 중... (dataset_dong-gu_gwangju_south_korea_10m_buf10m_baseline.gpkg)


추출된 피처 목록: ['nightlife_count_30m', 'nightlife_count_50m', 'nightlife_count_100m', 'cafe_count_30m', 'cafe_count_50m', 'cafe_count_100m', 'food_count_30m', 'food_count_50m', 'food_count_100m', 'retail_count_30m', 'retail_count_50m', 'retail_count_100m', 'service_count_30m', 'service_count_50m', 'service_count_100m']
Train shape: (90945, 15), Test shape: (22737, 15)
Train Positive: 1,424, Test Positive: 356



In [3]:
class PUBaggingXGBoost:
    def __init__(self, n_estimators=10, random_state=42, base_params=None):
        self.n_estimators = n_estimators
        self.models = []
        self.random_state = random_state
        self.base_params = base_params if base_params else {'n_estimators': 100, 'max_depth': 6, 'n_jobs': -1, 'eval_metric': 'logloss'}
        self.feature_names_ = None
        
    def fit(self, X_df, y):
        self.feature_names_ = list(X_df.columns)
        X = X_df.values
        p_idx = np.where(y == 1)[0]
        u_idx = np.where(y == 0)[0]
        
        rng = np.random.default_rng(self.random_state)
        for i in range(self.n_estimators):
            sampled_u_idx = rng.choice(u_idx, size=len(p_idx), replace=True)
            train_idx = np.concatenate([p_idx, sampled_u_idx])
            X_train = X[train_idx]
            y_train = np.concatenate([np.ones(len(p_idx)), np.zeros(len(sampled_u_idx))])
            
            model = xgb.XGBClassifier(random_state=self.random_state+i, **self.base_params)
            model.fit(X_train, y_train)
            self.models.append(model)
            
    def predict_proba(self, X_df):
        X = X_df[self.feature_names_].values
        probas = np.array([model.predict_proba(X)[:, 1] for model in self.models])
        return probas.mean(axis=0)

print("✅ PUBaggingXGBoost 클래스 준비 완료\n")


✅ PUBaggingXGBoost 클래스 준비 완료



In [4]:
from sklearn.metrics import recall_score, roc_auc_score
import numpy as np

model = PUBaggingXGBoost(n_estimators=10, random_state=42)
print("🏋️‍♂️ 모델 학습(Train)을 시작합니다... (이 작업은 다소 시간이 소요될 수 있습니다)")
model.fit(X_train, y_train)
print("✨ 모델 학습 완료!\n")

print("📊 PU-Learning 최적화 벤치마크 (Test Set):")
y_pred_proba = model.predict_proba(X_test)
y_pred = (y_pred_proba > 0.5).astype(int)

# PUF-score 계산 (Lee and Liu, 2003)
recall = recall_score(y_test, y_pred)
prob_positive = y_pred.mean() # 예측을 1로 한 비율 P[f(x)=1]
puf_score = (recall ** 2) / prob_positive if prob_positive > 0 else 0

print(f"🔹 Recall (재현율, r): {recall:.4f}")
print(f"🔹 P[f(x)=1] (Positive 예측 비율): {prob_positive:.4f}")
print(f"🏆 PUF-Score: {puf_score:.4f}")
print(f"📈 ROC AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}\n")

# 결과 텐서 검증 (Rule 5)
sample_scores = model.predict_proba(X_train.iloc[:1000])
sample_scores = np.nan_to_num(sample_scores, nan=0.0)
sample_scores = np.clip(sample_scores, 0.0, 1.0)
if sample_scores.min() >= 0.0 and sample_scores.max() <= 1.0:
    print("🟩 [PASS] Rule 5를 통과했습니다. 안전한 점수 분포입니다.\n")

model_save_path = model_dir / 'universal_xgb_model.pkl'
joblib.dump(model, model_save_path)
print(f"🚀 [완료] 학습된 모델이 성공적으로 저장되었습니다! -> {{model_save_path}}\n")


🏋️‍♂️ 모델 학습(Train)을 시작합니다... (이 작업은 다소 시간이 소요될 수 있습니다)


✨ 모델 학습 완료!

📊 PU-Learning 최적화 벤치마크 (Test Set):
🔹 Recall (재현율, r): 0.8371
🔹 P[f(x)=1] (Positive 예측 비율): 0.3257
🏆 PUF-Score: 2.1515
📈 ROC AUC Score: 0.8288

🟩 [PASS] Rule 5를 통과했습니다. 안전한 점수 분포입니다.

🚀 [완료] 학습된 모델이 성공적으로 저장되었습니다! -> {model_save_path}

